In [2]:
import openai
import requests
import json
import dotenv

dotenv.load_dotenv()

client = openai.OpenAI()

messages = []

In [3]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description":"https://nomad-movies-2.nomadcoders.workers.dev/movies API 호출을 통해 인기 영화를 가져옵니다.",
            "parameters":{
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description":"https://nomad-movies-2.nomadcoders.workers.dev/{id} API 호출을 통해 영화 정보를 가져옵니다.",
            "parameters":{
                "type": "object",
                "properties": {
                    "id" : {
                        "type": "integer",
                        "description": "영화의 id"
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description":"https://nomad-movies-2.nomadcoders.workers.dev/{id}/similar API 호출을 통해 유사한 영화를 조회합니다.",
            "parameters":{
                "type": "object",
                "properties": {
                    "id" : {
                        "type": "integer",
                        "description": "영화의 id"
                    }
                },
                "required": ["id"]
            }
        }
    }
]

In [4]:
def get_popular_movies():
    response = requests.get("https://nomad-movies-2.nomadcoders.workers.dev/movies")
    return response.json()

def get_movie_details(id):
    response = requests.get(f"https://nomad-movies-2.nomadcoders.workers.dev/movies/{id}")
    return response.json()

def get_similar_movies(id):
    response = requests.get(f"https://nomad-movies-2.nomadcoders.workers.dev/movies/{id}/similar")
    return response.json()

FUNCTION_MAP = {
    'get_popular_movies': get_popular_movies,
    'get_movie_details': get_movie_details,
    'get_similar_movies': get_similar_movies
}

In [6]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments
                        }
                    } for tool_call in message.tool_calls
                ]
        })

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(result)
            })

            print(f"Agent: [{function_name}({arguments}) 호출]")

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"Agent: {message.content}\n")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-5-nano", messages=messages, tools=TOOLS
    )
    process_ai_response(response.choices[0].message)

In [7]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()

User: 지금 인기 있는 영화 알려줘
Agent: [get_popular_movies({}) 호출]
Agent: 다음은 현재 인기 있는 영화 상위 5편입니다.

1) Obsession
- 언어: 영어 (en)
- 출시일: 2026-05-13
- 간단 소개: 한 소망을 이루면 원하는 사람의 마음을 얻지만, 그 욕망은 어두운 대가를 남긴다.
- 평점: 7.9
- 인기도: 852.33

2) Peddi
- 언어: 텔루구어 (te)
- 출시일: 2026-06-03
- 간단 소개: 1980년대 안드라 Pradesh의 시골에서 스포츠로 공동체를 하나로 묶는 이야기.
- 평점: 6.47
- 인기도: 836.46

3) Hai Jawani Toh Ishq Hona Hai
- 언어: 힌디어 (hi)
- 출시일: 2026-06-04
- 간단 소개: 결혼을 둘러싼 갈등과 새로운 사랑 사이에서 진정한 헌신의 의미를 찾아가는 이야기.
- 평점: 5.36
- 인기도: 553.84

4) The Unknown Man (L’homme inconnu)
- 언어: 네덜란드어/프랑스어 범주 (nl)
- 출시일: 2021-10-16
- 간단 소개: 벨기에 작가가 격리된 해변가에서 영감을 찾으려는 이야기.
- 평점: 8.20
- 인기도: 498.34

5) Michael
- 언어: 영어 (en)
- 출시일: 2026-04-22
- 간단 소개: 마이클 잭슨의 삶과 예술적 여정을 다룬 전기 영화.
- 평점: 8.51
- 인기도: 382.80

원하시면 상위 10편까지 확장해서 정리해 드리거나, 특정 영화의 상세 정보나 포스터/예고편 링크도 함께 알려드릴게요. 어떤 방식으로 더 보고 싶으신가요?

User: Michael 영화에 대해 더 알려줘
Agent: [get_movie_details({'id': 936075}) 호출]
Agent: 다음은 Michael 영화에 대한 자세한 정보입니다.

- 제목: Michael
- 원제/언어: English
- 개봉일: 2026-04-22
- 런타임: 128분
